# Benchmark Dataset and Formula Quality

Use this notebook after generating a synthetic benchmark dataset and before running the training pipeline. It checks formula homogeneity, formula complexity, positivity/finite behavior, and whether the generated stress samples satisfy the hidden unit-surface equation `f(sigma)=1`.

In [ ]:
from pathlib import Path
import sys
import json
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from invariant_generator.benchmark import (
    benchmark_root,
    evaluate_benchmark_dataset_quality,
    generate_benchmark_dataset,
)
from invariant_generator.config import load_config

CONFIG_PATH = PROJECT_ROOT / 'configs/self_eval_benchmark.toml'
CASE = 'single_H2'
RUN_GENERATION = False

config = load_config(CONFIG_PATH)
root = benchmark_root(config)
print('benchmark root:', root)
print('case:', CASE)

In [ ]:
if RUN_GENERATION:
    generated = generate_benchmark_dataset(config, CASE)
    print('generated dataset:', generated.dataset_path)

quality = evaluate_benchmark_dataset_quality(config, CASE)
quality_path = Path(quality['metadata_path']).with_name('formula_dataset_quality.json')
print('quality json:', quality_path)

In [ ]:
summary = {
    'formula': quality['formula_expression'],
    'family': quality['formula_family'],
    'difficulty': quality['difficulty'],
    'active_invariants': quality['active_invariants'],
    'complexity': quality['formula_complexity'],
    'homogeneity': quality['formula_homogeneity'],
    'dataset_surface': quality['dataset_surface'],
    'random_direction_values': quality['formula_value_random_directions'],
}
pprint(summary)

In [ ]:
surface = quality['dataset_surface']
hom = quality['formula_homogeneity']
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].bar(['surface max abs error', 'surface rel L2'], [
    surface['surface_value_max_abs_error'],
    surface['surface_value_relative_l2_error'],
])
axes[0].set_yscale('log')
axes[0].set_title('Unit-surface quality')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(['homogeneity rel L2', 'homogeneity max rel'], [
    hom['relative_l2_error'],
    hom['max_relative_error'],
])
axes[1].set_yscale('log')
axes[1].set_title('Formula homogeneity')
axes[1].tick_params(axis='x', rotation=20)

fig.tight_layout()

In [ ]:
stats = quality['invariant_statistics_on_surface']
names = list(stats)
stds = np.array([stats[name]['std'] for name in names], dtype=float)
means = np.array([stats[name]['mean'] for name in names], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(names, means)
axes[0].set_title('Homogenized invariant means on surface')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(names, stds)
axes[1].set_title('Homogenized invariant stds on surface')
axes[1].tick_params(axis='x', rotation=45)
fig.tight_layout()